# D-11 · ADN de Diseño (`tools/adn_diseno/colab.ipynb`)

Cuaderno autocontenido para ejecutar el estudio de **ADN de diseño** (paleta CIE $L^*a^*b^*$ + $\Delta E_{2000}$, retícula, vacío, jerarquía y saliencia visual profunda/espectral) a escala sobre un corpus grande en Google Colab con GPU, guardando los resultados `estudio.json` en Google Drive e imprimiendo el **SHA-256** de cada archivo procesado y generado.

**Garantías de licencia y privacidad:**
- Los referentes se estudian únicamente para extraer relaciones numéricas (`estudio.json`); no se copian ni republicarán píxeles.
- Todo color se expresa en CIE $L^*a^*b^*$ (D65, $2^\circ$) con distancias $\Delta E_{2000}$ (CIEDE2000, Sharma et al., 2005), nunca RGB crudo.

In [ ]:
# 1. Montaje de Google Drive y verificación de GPU
import hashlib
import json
import math
import os
import subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_OUT_DIR = Path('/content/drive/MyDrive/Prometeo_ADN_Diseno/estudios')
except ImportError:
    DRIVE_OUT_DIR = Path('./salidas_drive_local')

DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run(['nvidia-smi'], check=False)

In [ ]:
# 2. Clonado o actualización de tools/adn_diseno y ejecución por lotes con SHA-256 impreso
import numpy as np
from PIL import Image

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

print(f'Directorio de salida en Drive preparado: {DRIVE_OUT_DIR}')
manifiesto_lote = []

# Si el repositorio está presente en la sesión, importa estudiar_referente directamente
try:
    from tools.adn_diseno.estudiar import estudiar_referente
    CORPUS_DIR = Path(os.environ.get('ADN_CORPUS_DIR', '/content/drive/MyDrive/REFERENTES/corpus-editorial/muestras'))
    if CORPUS_DIR.is_dir():
        for img_path in sorted(CORPUS_DIR.glob('*.*')):
            if img_path.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.pdf'}:
                continue
            estudio = estudiar_referente(img_path, max_dim_analisis=256)
            out_json = DRIVE_OUT_DIR / f'{img_path.stem}.estudio.json'
            out_json.write_text(json.dumps(estudio, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
            sha_out = sha256_file(out_json)
            print(f'[OK] {img_path.name} -> {out_json.name} | sha256_origen={estudio["referente"]["sha256_origen"]} | sha256_estudio={sha_out}')
            manifiesto_lote.append({
                'archivo_origen': img_path.name,
                'sha256_origen': estudio['referente']['sha256_origen'],
                'archivo_estudio': out_json.name,
                'sha256_estudio': sha_out,
            })
        man_path = DRIVE_OUT_DIR / 'manifiesto_sha256.json'
        man_path.write_text(json.dumps(manifiesto_lote, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
        print(f'Manifiesto guardado en {man_path} (sha256={sha256_file(man_path)})')
except Exception as exc:
    print(f'Aviso de entorno Colab: {exc}')